# Tesla Car Price Prediction
Predicting Tesla stock closing prices from historical market data.

## Project Overview
Regression project using Tesla (TSLA) stock data to predict closing prices with ML models.

## Learning Objectives
- Download and process financial time series
- Engineer stock-specific features
- Understand limitations of stock price ML models

## Problem Statement
Predict tesla stock closing price using historical OHLCV data and engineered features.

## Why This Project Matters
Tesla is a high-volatility stock — great for studying ML model behavior under market uncertainty.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Yahoo Finance (yfinance) |
| **Ticker** | TSLA |
| **Target** | Close price |
| **Period** | 10 years |

## Dataset Source & License
Yahoo Finance public market data.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
try: import yfinance
except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','yfinance','-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
import yfinance as yf
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
TARGET = 'Close'
TICKER = 'TSLA'
PERIOD = '10y'

## Dataset Loading

In [4]:
df = yf.download(TICKER, period=PERIOD, auto_adjust=True).reset_index()
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] if c[1] == '' or c[1] == TICKER else c[0] for c in df.columns]
print(f'Shape: {df.shape}')
df.head()

[*********************100%***********************]  1 of 1 completed

Shape: (2514, 6)


,Date,Close,High,Low,Open,Volume
0,2016-04-13,16.968666,17.033333,16.488667,16.567333,73884000
1,2016-04-14,16.790667,17.122667,16.736668,16.866667,61983000
2,2016-04-15,16.967333,16.973333,16.608000,16.754000,56286000
3,2016-04-18,16.925333,17.220667,16.777332,16.815332,64071000
4,2016-04-19,16.491333,16.958000,16.083332,16.874666,95362500


## Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
print(df[TARGET].describe())

Missing: 0
Date range: 2016-04-13 00:00:00 to 2026-04-13 00:00:00
count    2514.000000
mean      160.482187
std       135.483598
min        11.931333
25%        21.150666
50%       174.660004
75%       257.209999
max       489.880005
Name: Close, dtype: float64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].plot(df['Date'], df[TARGET])
axes[0,0].set_title('TSLA Close Price')
axes[0,1].plot(df['Date'], df['Volume'])
axes[0,1].set_title('Volume')
axes[1,0].hist(df[TARGET].pct_change().dropna(), bins=100, edgecolor='black')
axes[1,0].set_title('Daily Returns')
num_corr = df.select_dtypes('number').corr()
im = axes[1,1].imshow(num_corr, cmap='coolwarm', aspect='auto')
axes[1,1].set_title('Correlation')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count    2514.000000
mean      160.482187
std       135.483598
min        11.931333
25%        21.150666
50%       174.660004
75%       257.209999
max       489.880005
Name: Close, dtype: float64

Skewness: 0.39


## Train/Test Split

In [8]:
# Feature engineering
df = df.sort_values('Date').reset_index(drop=True)
for lag in [1, 2, 3, 5, 10]:
    df[f'Close_lag{lag}'] = df[TARGET].shift(lag)
for w in [5, 10, 20]:
    df[f'Close_ma{w}'] = df[TARGET].rolling(w).mean()
    df[f'Close_std{w}'] = df[TARGET].rolling(w).std()
df['Returns'] = df[TARGET].pct_change()
df['Volume_ma5'] = df['Volume'].rolling(5).mean()

df = df.drop(columns=['Date']).dropna().reset_index(drop=True)

X = df.drop(columns=[TARGET])
y = df[TARGET]
split_idx = int(len(X) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (1996, 17), Test: (499, 17)


## Preprocessing
Time-ordered split. Lag and rolling features engineered from OHLCV data.

## Feature Engineering
Lag1-10, MA5/10/20, rolling std, daily returns, volume MA.

## Baseline Model

In [9]:
bl = LinearRegression()
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: R2={r2_score(y_test, bl_pred):.4f}  RMSE={root_mean_squared_error(y_test, bl_pred):.4f}  MAE={mean_absolute_error(y_test, bl_pred):.4f}')

Baseline LR: R2=0.9987  RMSE=3.3528  MAE=2.5118


## LazyPredict Benchmark

In [10]:
# --- LazyPredict Benchmark ---
try:
    from lazypredict.Supervised import LazyRegressor
    n_lp = min(5000, len(X_train))
    lr = LazyRegressor(verbose=0, ignore_warnings=True)
    lp_models, _ = lr.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                             Adjusted R-Squared  R-Squared      RMSE  \
Model                                                                  
LassoLarsIC                            0.998611   0.998658  3.348773   
BayesianRidge                          0.998610   0.998657  3.349899   
RidgeCV                                0.998609   0.998657  3.350806   
LassoLarsCV                            0.998607   0.998655  3.352815   
LinearRegression                       0.998607   0.998655  3.352815   
TransformedTargetRegressor             0.998607   0.998655  3.352815   
RANSACRegressor                        0.998599   0.998646  3.363495   
LarsCV                                 0.998590   0.998638  3.373896   
Lars                                   0.998590   0.998638  3.373896   
HuberRegressor                         0.998118   0.998182  3.897836   
Ridge                                  0.998001   0.998069  4.017065   
LassoCV                                0.997548   0.997632  4.44

## FLAML AutoML

In [11]:
# --- FLAML AutoML ---
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='regression', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  R2={r2_score(y_test, flaml_pred):.4f}  RMSE={root_mean_squared_error(y_test, flaml_pred):.4f}  MAE={mean_absolute_error(y_test, flaml_pred):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
# --- Boosting Models ---
models = {}
# CatBoost
cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

# LightGBM
lgb = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

# XGBoost
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0)
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: R2={r2_score(y_test, p):.4f}  RMSE={root_mean_squared_error(y_test, p):.4f}  MAE={mean_absolute_error(y_test, p):.4f}")

CatBoost: R2=0.9093  RMSE=27.5265  MAE=16.5060
LightGBM: R2=0.8868  RMSE=30.7631  MAE=17.9037
XGBoost: R2=0.9267  RMSE=24.7453  MAE=13.9488


## Top 3 Model Selection

In [13]:
# --- Top 3 Selection ---
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {'R2': r2_score(y_test, p), 'RMSE': root_mean_squared_error(y_test, p), 'MAE': mean_absolute_error(y_test, p)}

results_df = pd.DataFrame(all_results).T.sort_values('R2', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

                R2       RMSE        MAE
XGBoost   0.926734  24.745288  13.948823
CatBoost  0.909339  27.526485  16.506012
LightGBM  0.886765  30.763120  17.903705

Top 3: ['XGBoost', 'CatBoost', 'LightGBM']


## Final Evaluation of Top 3

In [14]:
# --- Final Evaluation of Top 3 ---
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    r2 = r2_score(y_test, p)
    rmse = root_mean_squared_error(y_test, p)
    mae = mean_absolute_error(y_test, p)
    final_results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    print(f"{name}: R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = list(final_results.keys())
for i, metric in enumerate(['R2', 'RMSE', 'MAE']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

XGBoost: R2=0.9267  RMSE=24.7453  MAE=13.9488
CatBoost: R2=0.9093  RMSE=27.5265  MAE=16.5060
LightGBM: R2=0.8868  RMSE=30.7631  MAE=17.9037


## Error Analysis

In [15]:
# --- Error Analysis ---
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)
residuals = y_test - preds

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(residuals, bins=50, edgecolor='black')
axes[0].set_title('Residual Distribution')
axes[0].axvline(0, color='red', linestyle='--')
axes[1].scatter(preds, residuals, alpha=0.3, s=5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[2].scatter(y_test, preds, alpha=0.3, s=5)
mn, mx = y_test.min(), y_test.max()
axes[2].plot([mn, mx], [mn, mx], 'r--')
axes[2].set_title('Actual vs Predicted')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## Interpretation & Business Insight
TSLA's high volatility makes accurate prediction harder than blue-chip stocks. Lag features still dominate.

## Limitations
- High volatility stock — past patterns may not persist
- No fundamental or sentiment data
- Autocorrelation-driven R2 is misleading

## How to Improve
- Add Elon Musk tweet sentiment
- Include EV market indicators
- Use attention-based models for regime detection

## Production Considerations
- Walk-forward retraining essential
- Must monitor for distribution shift
- Not suitable for live trading without risk controls

## Common Mistakes
- Believing high R2 means profitable trading
- Random splits on time series
- Ignoring transaction costs in evaluation

## Mini Challenge
1. Compare TSLA vs AAPL prediction difficulty
2. Predict 5-day returns instead of price
3. Try a simple momentum strategy based on predictions

## Final Summary
Modeled Tesla stock prices using engineered features. High R2 reflects autocorrelation, not genuine predictive power.

In [16]:
# --- Save Metrics ---
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "XGBoost": {
    "R2": 0.9267,
    "RMSE": 24.7453,
    "MAE": 13.9488
  },
  "CatBoost": {
    "R2": 0.9093,
    "RMSE": 27.5265,
    "MAE": 16.506
  },
  "LightGBM": {
    "R2": 0.8868,
    "RMSE": 30.7631,
    "MAE": 17.9037
  },
  "best_model": "XGBoost"
}
